In [ ]:
# 📦 Required installs
# !pip install langchain openai faiss-cpu

from langchain_classic.schema import Document
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

# 1. Load the text file
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 2. Split the text into chunks
splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_text(raw_text)
# Wrapping chunks as documents
documents = [Document(page_content=chunk) for chunk in chunks]

# 3. Create vector store
embedding = OllamaEmbeddings(
    model='nomic-embed-text'
)
# Storing everything in a searchable database
vectorstore = FAISS.from_documents(documents, embedding=embedding)

# 4. Setup LLM
llm = ChatOllama(
    model='mistral',
    temperature=0)


Created a chunk of size 622, which is longer than the specified 300
Created a chunk of size 803, which is longer than the specified 300


In [ ]:
#Get retrieves --> it does not generate
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# Compressor using LLM - Creating a “compressor” (AI filter)
compressor = LLMChainExtractor.from_llm(llm)

# Create compression retriever - Retrieval and Compression
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever()
)

# Run retrieval - Search document using meaning (via FAISS), Pass results to AI, AI removes irrelevant parts
print("🔹 Contextual Compression Results:")
results = compression_retriever.invoke("Who created LangChain?")
for doc in results:
    print("-", doc.page_content)


🔹 Contextual Compression Results:
- "LangChain was created by **Harrison Chase**."
- The Origin Story [No specific creator mentioned]
- Key Facts about the Creators:
(This part is relevant to answer the question)
- He launched it as an open-source project in October 2022 while he was working at a machine learning startup called Robust Intelligence.
Chase co-founded the company LangChain Inc. in early 2023 alongside Ankush Gola.


In [15]:
#Integration with ConversationalRetrievalChain- Without agent mode

from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever()
)

# ✅ Memory
memory = ConversationBufferMemory(
    return_messages=True, 
    memory_key="chat_history")

# ✅ Chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=compression_retriever,
    memory=memory
)

# 🧪 Ask a few questions
print("🔹 ConversationalRetrievalChain:")

print(rag_chain.invoke({"question": "What is LangChain?"}))
print(rag_chain.invoke({"question": "Who created it?"}))


🔹 ConversationalRetrievalChain:
{'question': 'What is LangChain?', 'chat_history': [HumanMessage(content='What is LangChain?', additional_kwargs={}, response_metadata={}), AIMessage(content=" Based on the provided context, LangChain appears to be an open-source framework designed to bridge the gap between static Language Models (LLMs) and dynamic, data-aware applications. It allows developers to 'chain' together different components such as prompt templates, memory modules, and document loaders to create sophisticated workflows. With LangChain, a project can implement Retrieval-Augmented Generation (RAG), where the LLM queries a private database or the web before generating an answer. However, without more specific context, it's not entirely clear what type of technology LangChain is exactly (e.g., a library, a platform, etc.).", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'answer': " Based on the provided context, LangChain appears to be an open

In [17]:
#Integration into Agent (with Tool)

from langchain_classic.tools import StructuredTool
from langchain_classic.agents import initialize_agent, AgentType

# 5. Wrap RAG as a StructuredTool
def rag_tool_fn(question: str) -> str:
    return rag_chain.invoke({
        "question": question
    })["answer"]

# Structured Tool
rag_tool = StructuredTool.from_function(
    name="RAG_Tool",
    description="Answer LangChain-related questions with context.",
    func=rag_tool_fn
)

# Agent with memory
agent = initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

# 🧪 Ask via agent
print("\n🔹 Agent Conversation:")
print(agent.invoke("What is LangChain?"))
print(agent.invoke("Who created it?"))



🔹 Agent Conversation:


> Entering new AgentExecutor chain...
 Thought: Do I need to use a tool? No
AI: Based on the provided context, LangChain appears to be an open-source framework designed to bridge the gap between static Language Models (LLMs) and dynamic, data-aware applications. It allows developers to 'chain' together different components such as prompt templates, memory modules, and document loaders to create sophisticated workflows. With LangChain, a project can implement Retrieval-Augmented Generation (RAG), where the LLM queries a private database or the web before generating an answer. However, without more specific context, it's not entirely clear what type of technology LangChain is exactly (e.g., a library, a platform, etc.).
(Note: The previous conversation history was provided to help provide a more accurate response.)

> Finished chain.
{'input': 'What is LangChain?', 'chat_history': [HumanMessage(content='What is LangChain?', additional_kwargs={}, response_metadata

In [19]:
#multiQuery

from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.chains import RetrievalQA


# MultiQueryRetriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# RetrievalQA Chain
rag_multi = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=multi_query_retriever,
    return_source_documents=True
)

# 🧪 Ask question
print("\n🔹 RAG Pipeline (MultiQueryRetriever):")
res = rag_multi.invoke({"query": "Tell me about LangChain creator and features in bullet points.."})

print("Answer:\n", res["result"])
print("\n\nSources:\n")
for doc in res["source_documents"]:
    print("-", doc.page_content[:200])  # print first 200 chars of each doc




🔹 RAG Pipeline (MultiQueryRetriever):
Answer:
  - Creator: Harrison Chase
- Project Launch: October 2022 (as an open-source project)
- Current Company: Co-founder of LangChain Inc. (early 2023)
- Previous Workplace: Machine learning startup Robust Intelligence
- Initial Development: Started as a side project on GitHub
- Problem Solved: Addresses the "plumbing" issues in LLMs, such as memory, PDF connection, and sequence of steps standardization
- Project Significance: Fastest-growing open-source project on GitHub at the time of launch
- Key Feature: LangChain is an open-source framework that bridges the gap between static LLMs and dynamic, data-aware applications
- Functionality: Allows developers to "chain" together different components to create sophisticated workflows, enabling Retrieval-Augmented Generation (RAG) for minimizing "hallucinations" and ensuring factual, up-to-date information in AI output.


Sources:

- LangChain was created by **Harrison Chase**.
- Key Facts about th